## One-Turn-Delayed Token Eviction

In [1]:
!pip install -q transformers accelerate

In [2]:
import torch
import math
from transformers.models.qwen2.modeling_qwen2 import apply_rotary_pos_emb


from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

MAX_NEW_TOKENS = 256

DEVICE = "cuda"
DTYPE = torch.bfloat16

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
).to(DEVICE)

model.eval()

print("Model:", type(model).__name__)
print("Device:", model.device)
print("Dtype:", model.dtype)

print("Layers:", model.config.num_hidden_layers)
print("Attention heads:", model.config.num_attention_heads)
print("KV heads:", model.config.num_key_value_heads)
print("Hidden size:", model.config.hidden_size)

head_dim = (
    model.config.hidden_size
    // model.config.num_attention_heads
)

print("Head dimension:", head_dim)

print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model: Qwen2ForCausalLM
Device: cuda:0
Dtype: torch.bfloat16
Layers: 24
Attention heads: 14
KV heads: 2
Hidden size: 896
Head dimension: 64
EOS token: '<|im_end|>'
EOS token ID: 151645


In [5]:
def get_kv_cache_bytes(past_key_values):
    key_bytes = 0
    value_bytes = 0

    for layer in past_key_values.layers:
        key_bytes += (
            layer.keys.numel()
            * layer.keys.element_size()
        )

        value_bytes += (
            layer.values.numel()
            * layer.values.element_size()
        )

    return key_bytes, value_bytes


def bytes_to_mib(num_bytes):
    return num_bytes / (1024 ** 2)


def print_cache_stats(past_key_values):
    key_bytes, value_bytes = get_kv_cache_bytes(
        past_key_values
    )

    total_bytes = key_bytes + value_bytes

    print(
        "KV tokens:",
        past_key_values.get_seq_length()
    )

    print(
        f"Key cache:   {bytes_to_mib(key_bytes):.4f} MiB"
    )

    print(
        f"Value cache: {bytes_to_mib(value_bytes):.4f} MiB"
    )

    print(
        f"Total KV:    {bytes_to_mib(total_bytes):.4f} MiB"
    )

In [6]:
def timed_cuda_call(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    torch.cuda.synchronize()

    start.record()

    result = fn()

    end.record()

    torch.cuda.synchronize()

    elapsed_ms = start.elapsed_time(end)

    return result, elapsed_ms

In [7]:
def generate_from_cache(
    outputs,
    past_key_values,
    generated_ids,
    attention_mask,
    max_new_tokens=MAX_NEW_TOKENS,
):

    assert (
        generated_ids.shape[1]
        == past_key_values.get_seq_length()
    )

    assert (
        attention_mask.shape[1]
        == past_key_values.get_seq_length()
    )

    next_token = torch.argmax(
        outputs.logits[:, -1, :],
        dim=-1,
        keepdim=True,
    )

    generated_token_ids = []
    decode_times_ms = []

    finished = False

    for _ in range(max_new_tokens):

        # --------------------------------
        # 1. Add predicted token
        # --------------------------------

        generated_ids = torch.cat(
            [generated_ids, next_token],
            dim=-1,
        )

        generated_token_ids.append(
            next_token.item()
        )

        # --------------------------------
        # 2. Extend attention mask by one
        # --------------------------------

        attention_mask = torch.cat(
            [
                attention_mask,
                torch.ones(
                    (attention_mask.shape[0], 1),
                    dtype=attention_mask.dtype,
                    device=attention_mask.device,
                ),
            ],
            dim=-1,
        )

        # --------------------------------
        # 3. Process new token into cache
        # --------------------------------

        def decode_step():
            with torch.no_grad():
                return model(
                    input_ids=next_token,
                    attention_mask=attention_mask,
                    past_key_values=past_key_values,
                    use_cache=True,
                )

        outputs, step_ms = timed_cuda_call(
            decode_step
        )

        decode_times_ms.append(step_ms)

        past_key_values = outputs.past_key_values

        # --------------------------------
        # 4. Correctness invariants
        # --------------------------------

        assert (
            generated_ids.shape[1]
            == past_key_values.get_seq_length()
        )

        assert (
            attention_mask.shape[1]
            == past_key_values.get_seq_length()
        )

        # --------------------------------
        # 5. Natural turn termination
        # --------------------------------

        if (
            next_token.item()
            == tokenizer.eos_token_id
        ):
            finished = True
            break

        # --------------------------------
        # 6. Predict following token
        # --------------------------------

        next_token = torch.argmax(
            outputs.logits[:, -1, :],
            dim=-1,
            keepdim=True,
        )

    return {
        "generated_ids": generated_ids,
        "past_key_values": past_key_values,
        "attention_mask": attention_mask,
        "outputs": outputs,
        "generated_token_ids": generated_token_ids,
        "decode_times_ms": decode_times_ms,
        "finished": finished,
    }

In [8]:
t1_messages = [
    {
        "role": "user",
        "content": (
            "I'm planning a tea party today. "
            "The theme is lavender and the special dessert "
            "is lemon cake. Remember these details for me."
        ),
    }
]

t1_text = tokenizer.apply_chat_template(
    t1_messages,
    tokenize=False,
    add_generation_prompt=True,
)

t1_inputs = tokenizer(
    t1_text,
    return_tensors="pt",
).to(model.device)

generated_ids = (
    t1_inputs["input_ids"].clone()
)

attention_mask = (
    t1_inputs["attention_mask"].clone()
)

print("TURN 1")
print()
print(t1_text)

print(
    "\nPrompt tokens:",
    generated_ids.shape[1]
)

TURN 1

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I'm planning a tea party today. The theme is lavender and the special dessert is lemon cake. Remember these details for me.<|im_end|>
<|im_start|>assistant


Prompt tokens: 55


In [9]:
def run_t1_prefill():

    with torch.no_grad():

        return model(
            input_ids=t1_inputs["input_ids"],
            attention_mask=attention_mask,
            use_cache=True,
        )


outputs, t1_prefill_ms = timed_cuda_call(
    run_t1_prefill
)

past_key_values = outputs.past_key_values


assert (
    generated_ids.shape[1]
    == past_key_values.get_seq_length()
)

assert (
    attention_mask.shape[1]
    == past_key_values.get_seq_length()
)


print(
    f"Turn 1 prefill: "
    f"{t1_prefill_ms:.3f} ms"
)

print()

print_cache_stats(
    past_key_values
)

Turn 1 prefill: 873.378 ms

KV tokens: 55
Key cache:   0.3223 MiB
Value cache: 0.3223 MiB
Total KV:    0.6445 MiB


In [10]:
t1_result = generate_from_cache(
    outputs=outputs,
    past_key_values=past_key_values,
    generated_ids=generated_ids,
    attention_mask=attention_mask,
)

generated_ids = (
    t1_result["generated_ids"]
)

past_key_values = (
    t1_result["past_key_values"]
)

attention_mask = (
    t1_result["attention_mask"]
)

outputs = (
    t1_result["outputs"]
)

t1_generated = (
    t1_result["generated_token_ids"]
)

t1_decode_times = (
    t1_result["decode_times_ms"]
)

t1_finished = (
    t1_result["finished"]
)


print(
    "Naturally terminated:",
    t1_finished
)

print(
    "Generated tokens:",
    len(t1_generated)
)

print(
    f"Total decode time: "
    f"{sum(t1_decode_times):.3f} ms"
)

print(
    f"Mean decode/token: "
    f"{sum(t1_decode_times) / len(t1_decode_times):.3f} ms"
)

print()

print_cache_stats(
    past_key_values
)

print("\nConversation:")

print(
    tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=False,
    )
)


assert t1_finished, (
    "Turn 1 hit MAX_NEW_TOKENS before EOS. "
    "Do not continue to Turn 2."
)

Naturally terminated: True
Generated tokens: 123
Total decode time: 3705.397 ms
Mean decode/token: 30.125 ms

KV tokens: 178
Key cache:   1.0430 MiB
Value cache: 1.0430 MiB
Total KV:    2.0859 MiB

Conversation:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I'm planning a tea party today. The theme is lavender and the special dessert is lemon cake. Remember these details for me.<|im_end|>
<|im_start|>assistant
Great! Here are the details for your tea party and lemon cake:

**Tea Party Details:**
- Date: [Insert Date]
- Time: [Insert Time]
- Location: [Insert Location]
- Guests: [Insert Guests]
- Theme: Lavender and Lemon
- Dessert: Lemon cake

**Lemon Cake Details:**
- Ingredients: [List of ingredients]
- Preparation: [Instructions for making the lemon cake]
- Serving: [Instructions for serving the lemon cake]

Please let me know if you need any additional details or if you have any specific requests for the party.<|i

In [11]:
layer_idx = 0

t1_end = past_key_values.get_seq_length()
t1_assistant_start = t1_inputs["input_ids"].shape[1]

captured_t2_queries = []
capture_queries = False

attn_module = model.model.layers[layer_idx].self_attn


def capture_post_rope_query(module, args, kwargs):
    if not capture_queries:
        return

    hidden_states = kwargs.get(
        "hidden_states",
        args[0] if len(args) > 0 else None,
    )

    position_embeddings = kwargs.get(
        "position_embeddings",
        args[1] if len(args) > 1 else None,
    )

    if position_embeddings is None:
        raise RuntimeError("Could not find position_embeddings in attention call.")

    q = module.q_proj(hidden_states)

    batch_size, seq_len, _ = q.shape
    num_q_heads = q.shape[-1] // module.head_dim

    q = q.view(
        batch_size,
        seq_len,
        num_q_heads,
        module.head_dim,
    ).transpose(1, 2)

    cos, sin = position_embeddings

    q_rotated, _ = apply_rotary_pos_emb(
        q,
        q,
        cos,
        sin,
    )

    captured_t2_queries.append(
        q_rotated.detach().clone()
    )


hook_handle = attn_module.register_forward_pre_hook(
    capture_post_rope_query,
    with_kwargs=True,
)

print("T1 cache length:", t1_end)
print("T1 assistant starts at:", t1_assistant_start)

T1 cache length: 178
T1 assistant starts at: 55


In [12]:
t2_user_text = "What theme and special dessert did I choose?"

t2_suffix = (
    "<|im_start|>user\n"
    + t2_user_text
    + "<|im_end|>\n"
    + "<|im_start|>assistant\n"
)

t2_inputs = tokenizer(
    t2_suffix,
    return_tensors="pt",
    add_special_tokens=False,
).to(model.device)

logical_position = t1_end

for i in range(t2_inputs["input_ids"].shape[1]):
    token = t2_inputs["input_ids"][:, i:i+1]

    attention_mask = torch.ones(
        (1, past_key_values.get_seq_length() + 1),
        dtype=torch.long,
        device=model.device,
    )

    position_id = torch.tensor(
        [[logical_position]],
        dtype=torch.long,
        device=model.device,
    )

    with torch.no_grad():
        outputs = model(
            input_ids=token,
            attention_mask=attention_mask,
            position_ids=position_id,
            past_key_values=past_key_values,
            use_cache=True,
        )

    past_key_values = outputs.past_key_values
    logical_position += 1

print("Cache after T2 suffix:", past_key_values.get_seq_length())

Cache after T2 suffix: 195


In [13]:
next_token = torch.argmax(
    outputs.logits[:, -1, :],
    dim=-1,
    keepdim=True,
)

generated_t2 = []

capture_queries = True

for _ in range(50):
    generated_t2.append(next_token.item())

    attention_mask = torch.ones(
        (1, past_key_values.get_seq_length() + 1),
        dtype=torch.long,
        device=model.device,
    )

    position_id = torch.tensor(
        [[logical_position]],
        dtype=torch.long,
        device=model.device,
    )

    with torch.no_grad():
        outputs = model(
            input_ids=next_token,
            attention_mask=attention_mask,
            position_ids=position_id,
            past_key_values=past_key_values,
            use_cache=True,
        )

    past_key_values = outputs.past_key_values
    logical_position += 1

    if next_token.item() == tokenizer.eos_token_id:
        break

    next_token = torch.argmax(
        outputs.logits[:, -1, :],
        dim=-1,
        keepdim=True,
    )

capture_queries = False
hook_handle.remove()

print(
    tokenizer.decode(
        generated_t2,
        skip_special_tokens=False,
    )
)

q_t2 = torch.cat(
    captured_t2_queries,
    dim=2,
)

print("Captured T2 Q shape:", q_t2.shape)

You have chosen a lavender theme and a lemon cake as the special dessert for your tea party. The lavender theme is inspired by the soft, delicate scent of lavender, which is often associated with relaxation and calmness. The lemon cake, on the other
Captured T2 Q shape: torch.Size([1, 14, 50, 64])


In [14]:
layer = past_key_values.layers[layer_idx]

k_t1 = layer.keys[
    :,
    :,
    t1_assistant_start:t1_end,
    :
]

print("T1 K shape:", k_t1.shape)

num_q_heads = q_t2.shape[1]
num_kv_heads = k_t1.shape[1]

group_size = num_q_heads // num_kv_heads
head_dim = q_t2.shape[-1]

scores_by_kv_head = []

for kv_head in range(num_kv_heads):
    q_start = kv_head * group_size
    q_end = q_start + group_size

    q_group = q_t2[
        :,
        q_start:q_end,
        :,
        :
    ]

    q_group = q_group.reshape(
        -1,
        head_dim,
    )

    k = k_t1[0, kv_head]

    logits = (
        q_group.float()
        @ k.float().T
    ) / math.sqrt(head_dim)

    attention = torch.softmax(
        logits,
        dim=-1,
    )

    scores = torch.sqrt(
        torch.mean(
            attention ** 2,
            dim=0,
        )
    )

    scores_by_kv_head.append(scores)

    print(
        f"KV head {kv_head}:",
        scores.shape,
    )

T1 K shape: torch.Size([1, 2, 123, 64])
KV head 0: torch.Size([123])
KV head 1: torch.Size([123])


In [15]:
assert len(captured_t2_queries) == len(generated_t2)

assert q_t2.shape[1] == model.config.num_attention_heads
assert q_t2.shape[-1] == head_dim

In [16]:
top_n = 15

for kv_head, scores in enumerate(scores_by_kv_head):
    top_local = torch.topk(
        scores,
        k=min(top_n, scores.numel()),
    ).indices

    top_global = (
        top_local
        + t1_assistant_start
    )

    print(f"\nKV head {kv_head}")

    for local_idx, global_idx in zip(
        top_local.tolist(),
        top_global.tolist(),
    ):
        token_id = generated_ids[0, global_idx].item()

        token_text = tokenizer.decode(
            [token_id],
            skip_special_tokens=False,
        )

        print(
            global_idx,
            f"{scores[local_idx].item():.6f}",
            repr(token_text),
        )


KV head 0
177 0.104031 '<|im_end|>'
117 0.098227 ' cake'
55 0.097226 'Great'
120 0.089145 'L'
113 0.085584 ' Dess'
98 0.081276 ' Guests'
110 0.077381 ' Lemon'
123 0.070547 ' Details'
94 0.069254 'Insert'
57 0.069239 ' Here'
102 0.068319 ' Guests'
105 0.068191 ' Theme'
127 0.065299 ' Ingredients'
114 0.064230 'ert'
60 0.061667 ' details'

KV head 1
176 0.127644 '.'
65 0.092479 ' and'
177 0.090690 '<|im_end|>'
136 0.083646 ':'
128 0.083351 ':'
124 0.082603 ':'
147 0.082038 ':'
56 0.081995 '!'
152 0.077332 ' the'
134 0.061576 '-'
74 0.061561 ':'
173 0.061043 ' for'
59 0.060686 ' the'
68 0.058605 ':\n\n'
61 0.058174 ' for'
